# v0.7



# predict => submission 

## Public Score: 1.68273






In [1]:
!mkdir -p content/my_dataset
!unzip -q umud-challenge-muscle-architecture-in-ultrasound-data.zip -d content/my_dataset/

"""
import sys
sys.path.append("/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data")
!pip uninstall -y -q torch torchvision torchaudio > /dev/null 2>&1
!pip install -q --no-cache-dir \
    torch==2.10.0 \
    torchvision==0.25.0 \
    torchaudio==2.10.0 \
    --index-url https://download.pytorch.org/whl/cu126 \
    > /dev/null 2>&1
"""

'\nimport sys\nsys.path.append("/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data")\n!pip uninstall -y -q torch torchvision torchaudio > /dev/null 2>&1\n!pip install -q --no-cache-dir     torch==2.10.0     torchvision==0.25.0     torchaudio==2.10.0     --index-url https://download.pytorch.org/whl/cu126     > /dev/null 2>&1\n'

In [1]:
from PIL import Image
import matplotlib.pyplot as plt
import os
import numpy as np
import torch
import torch.nn as nn
from scipy.ndimage import label

def decode_hardware_scale(img_np, filename):
    h, w = img_np.shape[:2]
    filetype = filename.split(".")[-1].lower()
    px_per_cm = -1.0
    l, t, r, b = -1, -1, -1, -1

    if filetype == "png":
        col6 = img_np[:, 6].mean(axis=-1) if img_np.ndim == 3 else img_np[:, 6]
        col9 = img_np[:, 9].mean(axis=-1) if img_np.ndim == 3 else img_np[:, 9]
        first_tick = int(np.argmax(col6 > 50))
        sec_minor = 150 + int(np.argmax(col6[150:] > 50))
        sec_major = 150 + int(np.argmax(col9[150:] > 50))
        last_tick = len(img_np) - 1 - int(np.argmax(col6[::-1] > 50))

        if (sec_major - first_tick) < 3 * (sec_minor - first_tick):
            px_per_cm = float(sec_major - first_tick)
        else:
            px_per_cm = float(sec_minor - first_tick)

        hw = w // 2
        s = img_np[:, hw:].sum(axis=(0, 2)) if img_np.ndim == 3 else img_np[:, hw:].sum(axis=0)
        w2 = int(np.argmin(s))
        l, t, r, b = hw - w2, first_tick, hw + w2, last_tick

    elif h == 800 and w == 1200:
        is_right = False

        if img_np.ndim == 3 and ((img_np[87, 1147:1157] == 175).all() or img_np[87, 1147:1157].mean() > 150):
            is_right = True
        elif img_np.ndim == 2 and ((img_np[87, 1147:1157] == 175).all() or img_np[87, 1147:1157].mean() > 150):
            is_right = True

        if is_right:
            col1150 = img_np[:, 1150].mean(axis=-1) if img_np.ndim == 3 else img_np[:, 1150]
            first_tick = int(np.argmax(col1150 > 50))
            sec_major = first_tick + 20 + int(np.argmax(col1150[first_tick + 20:] > 50))
            last_tick = len(img_np) - 1 - int(np.argmax(col1150[::-1] > 50))
            denom = max(1, (sec_major - first_tick))
            n_ticks = int(round((last_tick - first_tick) / denom))

            table = {
                7: (142, 91, 1058), 8: (163, 91, 1037), 9: (211, 91, 989),
                10: (249, 91, 951), 11: (282, 91, 918), 12: (308, 91, 892),
                13: (331, 91, 869), 14: (349, 91, 851), 15: (142, 91, 1058)
            }

            if n_ticks <= 14:
                px_per_cm = (last_tick - first_tick) / max(1, n_ticks) * 2.0
            else:
                px_per_cm = (last_tick - first_tick) / 3.0

            if n_ticks in table:
                l, t, r = table[n_ticks]
                b = last_tick
            else:
                l, t, r, b = 142, 91, 1058, last_tick

        else:
            is_left = False

            if img_np.ndim == 3 and (img_np[42, 67:74, 0] > 115).all():
                is_left = True
            elif img_np.ndim == 2 and (img_np[42, 67:74] > 115).all():
                is_left = True

            if is_left:
                px_per_cm = (783.0 - 42.0) / 5.0
                l, t, r, b = 171, 42, 1029, 798
            else:
                px_per_cm = 148.2
                l, t, r, b = 171, 42, 1029, 798

    elif h == 644 and w == 1088:
        px_per_cm = 630.5 / 5.0
        l, t, r, b = 140, 0, 947, 643

    elif h in [512, 513]:
        px_per_cm = (442.0 - 53.0) / 5.0
        l, t, r, b = 0, 0, w, h - 10

    elif h == 853:
        if img_np.ndim == 2 and img_np[-5, 100] == 170 and img_np[-5, 934] == 170:
            px_per_cm = (934.0 - 100.0) / 5.0
        elif img_np.ndim == 2 and img_np[-5, 44] == 170 and img_np[-5, 879] == 170:
            px_per_cm = (879.0 - 44.0) / 5.0
        else:
            px_per_cm = (934.0 - 100.0) / 5.0

        l, t, r, b = 0, 0, w, h

    else:
        px_per_cm = 148.2
        l, t, r, b = 0, 0, w, h

    return float(max(10.0, px_per_cm)), int(l), int(t), int(r), int(b)


# 設定

#apo_model_path = "/kaggle/input/datasets/nagatakengo/segmentation-baseline-models/apo_model_pos_weight_1.5.pth"
#fasc_model_path = "/kaggle/input/datasets/nagatakengo/segmentation-baseline-models/fasc_model_pos_weight_15.pth"

apo_model_path = "segmentation_baseline_models/apo_model.pth"
fasc_model_path = "segmentation_baseline_models/fasc_model.pth"

apo_threshold = 0.5
fasc_threshold = 0.58

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)


# U-Net

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        # 出力
        self.output = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))
        enc4 = self.enc4(self.pool3(enc3))

        bottleneck = self.bottleneck(self.pool4(enc4))

        dec4 = self.up4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.up3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.up2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.up1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.dec1(dec1)

        return self.output(dec1)


def load_model(model_path):
    """
    model 読み込み
    """
    model = UNet()
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()

    return model

apo_model = load_model(apo_model_path)
fasc_model = load_model(fasc_model_path)

print("=======================================================")
print("apo model をロードしました...")
print("fasc model をロードしました...")
print("=======================================================")


def resize_image_with_padding(image, target_size=(768, 512)):
    """
    リサイズ（学習時と同じ 768 x 512）
    """
    target_width, target_height = target_size
    original_width, original_height = image.size

    scale = min(target_width / original_width, target_height / original_height)
    new_width = int(original_width * scale)
    new_height = int(original_height * scale)

    image = image.resize((new_width, new_height), resample=Image.Resampling.BILINEAR)

    left = (target_width - new_width) // 2
    top = (target_height - new_height) // 2

    image_canvas = Image.new(image.mode, target_size, 0)
    image_canvas.paste(image, (left, top))

    return image_canvas, scale


def predict_mask(model, image_path, threshold):
    """
    apo / fasc 共通予測
    """
    image = Image.open(image_path).convert("RGB")
    original_image_array = np.array(image)

    px_per_cm, l, t, r, b = decode_hardware_scale(
        original_image_array,
        os.path.basename(image_path)
    )

    image, resize_scale = resize_image_with_padding(image, target_size=(768, 512))
    image_array = np.array(image)
    image_tensor = torch.from_numpy(image_array).float() / 255.0
    image_tensor = image_tensor.permute(2, 0, 1)
    image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.to(device)

    with torch.no_grad():
        outputs = model(image_tensor)
        pred_probs = torch.sigmoid(outputs)
        pred_mask = (pred_probs >= threshold).float()

    pred_mask = pred_mask[0, 0].cpu().numpy()

    # geometryはリサイズ後画像上で計算するためscaleを反映
    resized_px_per_cm = px_per_cm * resize_scale

    return image_array, pred_mask, resized_px_per_cm


def extract_apo_lines(pred_mask):
    """
    superficial / deep apo 抽出
    """
    height, width = pred_mask.shape

    labeled_mask, num_components = label(pred_mask)

    superficial_component = None
    deep_component = None

    superficial_area = 0
    deep_area = 0

    for component_id in range(1, num_components + 1):
        component_mask = (labeled_mask == component_id)

        area = component_mask.sum()

        if area == 0:
            continue

        y_indices, x_indices = np.where(component_mask)
        mean_y = y_indices.mean()

        # 上半分 → superficial候補
        if mean_y < height / 2:
            if area > superficial_area:
                superficial_area = area
                superficial_component = component_mask

        # 下半分 → deep候補
        else:
            if area > deep_area:
                deep_area = area
                deep_component = component_mask

    superficial_y = np.full(width, np.nan)
    deep_y = np.full(width, np.nan)

    if superficial_component is not None:
        for x in range(width):
            y_indices = np.where(superficial_component[:, x])[0]

            if len(y_indices) > 0:
                superficial_y[x] = np.median(y_indices)

    if deep_component is not None:
        for x in range(width):
            y_indices = np.where(deep_component[:, x])[0]

            if len(y_indices) > 0:
                deep_y[x] = np.median(y_indices)

    return superficial_y, deep_y

def extract_fasc_lines(pred_mask, min_area=20, min_width=30):
    """
    fascicle line 抽出

    提案ベースライン：
    連結成分ごとに直線をfitする
    """
    labeled_mask, num_components = label(pred_mask)

    fasc_lines = []

    for component_id in range(1, num_components + 1):
        component_mask = (labeled_mask == component_id)

        area = component_mask.sum()

        # 小さい成分を除外
        if area < min_area:
            continue

        y_indices, x_indices = np.where(component_mask)

        if len(np.unique(x_indices)) < 2:
            continue

        x_width = x_indices.max() - x_indices.min()

        if x_width < min_width:
            continue

        slope, intercept = np.polyfit(x_indices, y_indices, 1)

        x1 = x_indices.min()
        x2 = x_indices.max()

        y1 = slope * x1 + intercept
        y2 = slope * x2 + intercept

        fasc_lines.append((x1, y1, x2, y2))

    return fasc_lines

def calculate_pa(deep_y, fasc_lines):
    """
    PA計算

    提案ベースライン：
    deep apoを直線近似し、
    各fascicle lineとの角度を計算する
    """
    x = np.arange(len(deep_y))
    valid = ~np.isnan(deep_y)

    if valid.sum() < 2:
        return []

    if len(np.unique(x[valid])) < 2:
        return []

    deep_slope, deep_intercept = np.polyfit(x[valid], deep_y[valid], 1)

    pa_values = []

    for x1, y1, x2, y2 in fasc_lines:
        fasc_vector = np.array([x2 - x1, y2 - y1])
        deep_vector = np.array([1.0, deep_slope])

        cos_angle = np.dot(fasc_vector, deep_vector)
        cos_angle /= np.linalg.norm(fasc_vector) * np.linalg.norm(deep_vector)

        cos_angle = np.clip(np.abs(cos_angle), 0.0, 1.0)

        pa_deg = np.degrees(np.arccos(cos_angle))
        pa_values.append(pa_deg)

    return pa_values

def calculate_fl(superficial_y, deep_y, fasc_lines, px_per_cm):
    """
    FL計算

    提案ベースライン：
    superficial / deep apoを直線近似し、
    fascicle lineを延長して両apoとの交点間距離を計算する
    """
    x = np.arange(len(superficial_y))

    superficial_valid = ~np.isnan(superficial_y)
    deep_valid = ~np.isnan(deep_y)

    if superficial_valid.sum() < 2:
        return []

    if deep_valid.sum() < 2:
        return []

    if len(np.unique(x[superficial_valid])) < 2:
        return []

    if len(np.unique(x[deep_valid])) < 2:
        return []

    superficial_slope, superficial_intercept = np.polyfit(
        x[superficial_valid], superficial_y[superficial_valid], 1
    )

    deep_slope, deep_intercept = np.polyfit(
        x[deep_valid], deep_y[deep_valid], 1
    )

    fl_values = []

    for x1, y1, x2, y2 in fasc_lines:
        if x2 == x1:
            continue

        fasc_slope = (y2 - y1) / (x2 - x1)
        fasc_intercept = y1 - fasc_slope * x1

        if np.isclose(fasc_slope, superficial_slope):
            continue

        if np.isclose(fasc_slope, deep_slope):
            continue

        superficial_x = (superficial_intercept - fasc_intercept) / (fasc_slope - superficial_slope)
        superficial_y_intersection = fasc_slope * superficial_x + fasc_intercept

        deep_x = (deep_intercept - fasc_intercept) / (fasc_slope - deep_slope)
        deep_y_intersection = fasc_slope * deep_x + fasc_intercept

        fl_px = np.sqrt(
            (deep_x - superficial_x) ** 2 +
            (deep_y_intersection - superficial_y_intersection) ** 2
        )

        fl_mm = fl_px * (10.0 / px_per_cm)

        if fl_mm < 30 or fl_mm > 200:
            continue

        fl_values.append(fl_px)

    return fl_values


def calculate_mt(superficial_y, deep_y):
    """
    MT計算

    提案ベースライン：
    deep apoを直線近似し、
    superficial apo上の各点からdeep apoへの垂直距離を計算する
    """
    x = np.arange(len(deep_y))
    deep_valid = ~np.isnan(deep_y)

    if deep_valid.sum() < 2:
        return []

    if len(np.unique(x[deep_valid])) < 2:
        return []

    deep_slope, deep_intercept = np.polyfit(x[deep_valid], deep_y[deep_valid], 1)

    mt_values = []

    for x_point, superficial_point in enumerate(superficial_y):
        if np.isnan(superficial_point):
            continue

        mt_px = abs(
            deep_slope * x_point - superficial_point + deep_intercept
        ) / np.sqrt(deep_slope ** 2 + 1)

        mt_values.append(mt_px)

    return mt_values


# testデータ全件予測 → submission.csv

import pandas as pd

test_dir = "content/my_dataset/test_images_v2/test_set_v2"
#test_dir = "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data/test_images_v2/test_set_v2"

test_files = sorted([
    f for f in os.listdir(test_dir)
    if f.lower().endswith((".tif", ".png"))
])

submission_rows = []

for i, filename in enumerate(test_files):
    image_path = os.path.join(test_dir, filename)

    # apo
    image, apo_pred_mask, px_per_cm = predict_mask(apo_model, image_path, apo_threshold)
    superficial_y, deep_y = extract_apo_lines(apo_pred_mask)

    # fasc
    _, fasc_pred_mask, _ = predict_mask(fasc_model, image_path, fasc_threshold)
    fasc_lines = extract_fasc_lines(fasc_pred_mask)

    # PA
    pa_values = calculate_pa(deep_y, fasc_lines)

    # FL
    fl_values = calculate_fl(    
        superficial_y,
        deep_y,
        fasc_lines,
        px_per_cm
    )

    # MT
    mt_values = calculate_mt(superficial_y, deep_y)

    # pixel → mm
    mm_per_px = 10.0 / px_per_cm

    fl_mm_values = [fl_px * mm_per_px for fl_px in fl_values]
    mt_mm_values = [mt_px * mm_per_px for mt_px in mt_values]

    # 1画像につき1値
    pa_deg = np.mean(pa_values)
    fl_mm = np.mean(fl_mm_values)
    mt_mm = np.mean(mt_mm_values)

    submission_rows.append({
        "image_id": filename,
        "pa_deg": pa_deg,
        "fl_mm": fl_mm,
        "mt_mm": mt_mm
    })

submission = pd.DataFrame(submission_rows)

# NaN補完
pa_median = submission["pa_deg"].median()
fl_median = submission["fl_mm"].median()
mt_median = submission["mt_mm"].median() 

submission["pa_deg"] = submission["pa_deg"].fillna(pa_median)
submission["fl_mm"] = submission["fl_mm"].fillna(fl_median)
submission["mt_mm"] = submission["mt_mm"].fillna(mt_median) 

print("NaN count:")
print(submission[["pa_deg", "fl_mm", "mt_mm"]].isna().sum())

print("NaN rows:")
print(submission[submission[["pa_deg", "fl_mm", "mt_mm"]].isna().any(axis=1)])


submission.to_csv(
    "submission.csv",
    index=False
)

print("=======================================================")
print("submission.csv を作成しました")
print("rows:", len(submission))
print("=======================================================")

print(submission.head())

device: cuda
apo model をロードしました...
fasc model をロードしました...


/home/nk/kengo/Muscle_Architecture_in_Ultrasound_Data/.venv/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/nk/kengo/Muscle_Architecture_in_Ultrasound_Data/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


NaN count:
pa_deg    0
fl_mm     0
mt_mm     0
dtype: int64
NaN rows:
Empty DataFrame
Columns: [image_id, pa_deg, fl_mm, mt_mm]
Index: []
submission.csv を作成しました
rows: 309
        image_id     pa_deg       fl_mm      mt_mm
0  IMG_00001.tif  13.389062  105.557539  25.043179
1  IMG_00002.tif   8.300126  113.647308  19.269759
2  IMG_00003.tif  10.577948  151.856603  24.119860
3  IMG_00004.tif  11.035910  196.754635  28.116627
4  IMG_00005.tif  16.578369  121.011913  30.290398
